In [1]:
from enum import Enum
import pandas as pd



class PrintingStates(Enum):
    """
    Defining enums for printing States
    """
    IDLE = 0
    IN_PROGRESS = 1
    PAUSED = 2
    ABORTED = 3
    FAILED = 4
    COMPLETED = 5

class FilamentReplacementStates(Enum):
    """
    Defining enums for Filament Replacement States
    """
    IDLE = 0
    MOVING_TO_WIPING_LOCATION = 1
    WIPING = 2
    NOZZLE_COOLING = 3
    LOADING_FILAMENT = 4
    NOZZLE_HEATING = 5







In [6]:
DONT_CARE = "X"
STATE_COLS = ["printing_state", "filament_change_state", "retries_left"]
INPUT_COLS = ["external_input", "system_feedback"]
OUTPUT_COLS = ["output_printing_state","output_filament_change_state","output_message","output_retries_left"]


transition_table = pd.read_csv(
    "state_transition_tables/filament_change_state_transition_table.csv",
    dtype=str, keep_default_na=False,  # keep "null", "X" and blanks as literal text
)

transition_table = transition_table.fillna("").apply(lambda col: col.str.strip())

def cell_matches(table_value, actual_value):
    if table_value == DONT_CARE:
        return True
    return table_value == actual_value

def resolve_output(table_value, current_value):
    return current_value if table_value in (DONT_CARE, "") else table_value

def state_update(current_state, system_input):
    print(f"CURRENT STATE: {current_state}")
    row_to_check = {
        "printing_state": current_state["printing_state"],
        "filament_change_state": current_state["filament_change_state"],
        "retries_left": current_state["retries_left"],
        "external_input": system_input["external_input"],
        "system_feedback": system_input["system_feedback"]
    }
    row_to_check = {k: "" if v is None else str(v).strip() for k, v in row_to_check.items()}

    def generate_output(input_row, table_row):
        for col in INPUT_COLS+STATE_COLS:

            if table_row[col] == DONT_CARE:
                continue
            elif table_row[col] != input_row[col]:
                return False, {}


        for col in OUTPUT_COLS:
            if table_row[col] == DONT_CARE:
                table_row[col] = input_row[col.removeprefix("output_")]
        output_state = {col: table_row[col] for col in OUTPUT_COLS}
        return True, output_state

    print(row_to_check)
    for _, row in transition_table.iterrows():
        row_match,output = generate_output(row_to_check,row)
        if row_match:
            next_state = {col:output[f"output_{col}"] for col in STATE_COLS}
            message = output["output_message"]

            return next_state,message
                

    return current_state, "NO MATCHING TRANSITION"



In [ ]:
current_state = {
    "printing_state": PrintingStates.IN_PROGRESS.name,
    "filament_change_state": FilamentReplacementStates.IDLE.name,
    "retries_left": 3
}

inputs = [
    {"external_input": "C", "system_feedback": None},
]
for system_input in inputs:
    current_state, msg = state_update(current_state, system_input)
    print(f"OUTPUT STATE: {current_state}")
    print(f"MESSAGE: {msg}")


CURRENT STATE: {'printing_state': 'IN_PROGRESS', 'filament_change_state': 'IDLE', 'retries_left': 3}
{'printing_state': 'IN_PROGRESS', 'filament_change_state': 'IDLE', 'retries_left': '3', 'external_input': 'C', 'system_feedback': ''}
OUTPUT STATE: {'printing_state': 'PAUSED', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}


In [4]:
import io
import contextlib

test_cases = pd.read_csv("state_transition_tables/test_cases.csv", dtype=str, keep_default_na=False)

results = []
for _, case in test_cases.iterrows():
    state = {
        "printing_state": case["printing_state"],
        "filament_change_state": case["filament_change_state"],
        "retries_left": case["retries_left"],
    }
    system_input = {
        "external_input": case["external_input"] or None,
        "system_feedback": case["system_feedback"] or None,
    }
    with contextlib.redirect_stdout(io.StringIO()):
        next_state, msg = state_update(state, system_input)

    handled = msg != "NO MATCHING TRANSITION"
    results.append({
        "case": case["case"],
        "description": case["description"],
        "realistic": case["realistic"],
        "handled": handled,
        "next_state": next_state if handled else "",
        "message": msg,
    })

results = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
display(results)

missing = results[(results["realistic"] == "yes") & ~results["handled"]]
print(f"\n{len(missing)} realistic case(s) have NO transition in the table:")
for _, r in missing.iterrows():
    print(f"  case {r['case']}: {r['description']}")

,case,description,realistic,handled,next_state,message
0,1,Start filament change (C) while printer idle,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
1,2,Filament runout (F) while printer idle,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
2,3,Filament change (C) requested mid-print,yes,True,"{'printing_state': 'PAUSED', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
3,4,Filament runout (F) mid-print,yes,True,"{'printing_state': 'PAUSED', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
4,5,Filament change (C) while print is paused,yes,True,"{'printing_state': 'PAUSED', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
5,6,Filament runout (F) while print is paused,yes,True,"{'printing_state': 'PAUSED', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '3'}","""FILAMENT CHANGE IN PROGRESS"""
6,11,Moving to wiping location succeeds first try,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'WIPING', 'retries_left': '3'}","""MOVING_TO_WIPING_LOCATION_SUCCESSFUL"""
7,12,Moving to wiping location fails first try,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'MOVING_TO_WIPING_LOCATION', 'retries_left': '2'}","""MOVING_TO_WIPING_LOCATION_FAILURE. TRYING AGAIN"""
8,13,Moving to wiping location fails with no retries left,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'IDLE', 'retries_left': '3'}",""" FILAMENT CHANGE PROCESS FAILED. INITIATE FILAMENT CHANGE AGAIN"""
9,14,Moving to wiping location succeeds after a retry,yes,True,"{'printing_state': 'IDLE', 'filament_change_state': 'WIPING', 'retries_left': '3'}","""MOVING_TO_WIPING_LOCATION_SUCCESSFUL"""



0 realistic case(s) have NO transition in the table:
